# AlphaHVAC — Root Cause Fixed
The orange line was zero because the damper drifts to 0 from random actions.
The policy loss was stuck at log(3)=1.099 meaning pure random policy.
This notebook fixes the root cause: reward shaping that creates strong contrast between actions.

In [1]:
# ── CELL 1: HONEST DIAGNOSIS ────────────────────────────────
import numpy as np

print("=" * 60)
print("HONEST DIAGNOSIS OF PREVIOUS MODEL")
print("=" * 60)
print()
print(f"log(3) = {np.log(3):.4f}  <-- entropy of a perfectly RANDOM policy")
print(f"Your policy_loss was: ~1.089 to 1.094  <-- nearly identical to log(3)")
print()
print("WHAT THIS MEANS:")
print("  The policy head learned to output [0.33, 0.33, 0.33] for every")
print("  state. It is completely random. It learned nothing.")
print()
print("WHY THE ORANGE LINE WAS ZERO:")
print("  Random policy = ~33% decrease, 33% hold, 33% increase.")
print("  damper_step = 0.025. Over 1724 test steps with random actions:")
print("  expected net change = 0.025*(decrease - increase) per step")
print("  But damper is CLIPPED at 0. Once it hits 0 it stays at 0.")
print("  damper=0 → airflow=0 → energy=0 → orange line flat at zero.")
print("  This is NOT energy saving. It is the damper stuck closed.")
print()
print("VALUE HEAD WAS FINE:")
print("  value_loss went from 0.019 to 0.008 — converging.")
print("  Value(ideal) - Value(crisis) = +0.4044 — good separation.")
print("  The value head learned. The policy head did not.")
print()
print("ROOT CAUSE:")
print("  MCTS visit distribution was always [0.33, 0.33, 0.33]")
print("  because the value head could not distinguish which action")
print("  leads to a better NEXT state within 20-100 simulations.")
print("  All 3 actions give similar 1-step rewards so MCTS sees no signal.")
print()
print("THE FIX:")
print("  Reshape the reward to create STRONG contrast between actions.")
print("  Add a large BONUS for correctly reducing damper when room is OK.")
print("  Add a large PENALTY for reducing damper when room needs cooling/heating.")
print("  This gives MCTS a clear signal within just 1 step.")
print("=" * 60)


HONEST DIAGNOSIS OF PREVIOUS MODEL

log(3) = 1.0986  <-- entropy of a perfectly RANDOM policy
Your policy_loss was: ~1.089 to 1.094  <-- nearly identical to log(3)

WHAT THIS MEANS:
  The policy head learned to output [0.33, 0.33, 0.33] for every
  state. It is completely random. It learned nothing.

WHY THE ORANGE LINE WAS ZERO:
  Random policy = ~33% decrease, 33% hold, 33% increase.
  damper_step = 0.025. Over 1724 test steps with random actions:
  expected net change = 0.025*(decrease - increase) per step
  But damper is CLIPPED at 0. Once it hits 0 it stays at 0.
  damper=0 → airflow=0 → energy=0 → orange line flat at zero.
  This is NOT energy saving. It is the damper stuck closed.

VALUE HEAD WAS FINE:
  value_loss went from 0.019 to 0.008 — converging.
  Value(ideal) - Value(crisis) = +0.4044 — good separation.
  The value head learned. The policy head did not.

ROOT CAUSE:
  MCTS visit distribution was always [0.33, 0.33, 0.33]
  because the value head could not distinguish wh

In [2]:
# ── CELL 2: DATA PREPROCESSING ──────────────────────────────
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

df = pd.read_csv("/Users/pawanpahune/AIFA Project/Dataset/B90_102_exp30m_202104.csv")
df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time").reset_index(drop=True)

cols = ["time","room_temp","thermostat_outside_temp","damper_position",
        "airflow_current","supply_discharge_temp","clg_signal","htg_signal",
        "htg_valve_position","clg_sp_current","htg_sp_current","htg_clg_mode"]
df = df[cols].copy()

df["hour_of_day"]    = df["time"].dt.hour / 23.0
df["day_of_week"]    = df["time"].dt.dayofweek / 6.0
df["setpoint"]       = df["htg_clg_mode"]*df["htg_sp_current"] + (1-df["htg_clg_mode"])*df["clg_sp_current"]
df["thermal_signal"] = df["htg_clg_mode"]*df["htg_signal"]     + (1-df["htg_clg_mode"])*df["clg_signal"]
df = df.drop(columns=["clg_sp_current","htg_sp_current"])
df["room_temp_lag1"] = df["room_temp"].shift(1)
df["damper_lag1"]    = df["damper_position"].shift(1)

num_cols = ["room_temp","thermostat_outside_temp","damper_position",
            "airflow_current","supply_discharge_temp","clg_signal","htg_signal",
            "htg_valve_position","setpoint","thermal_signal","room_temp_lag1","damper_lag1"]
scaler = RobustScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])
df[num_cols] = df[num_cols].clip(-3, 3)
df = df.set_index("time").dropna().sort_index()

df.to_csv("Dataset/Transformed_Optimized.csv", index=False)
df_full   = pd.read_csv("Dataset/Transformed_Optimized.csv")
split_idx = int(len(df_full) * 0.7)
df_full.iloc[:split_idx].reset_index(drop=True).to_csv("Dataset/Train_Optimized.csv", index=False)
df_full.iloc[split_idx:].reset_index(drop=True).to_csv("Dataset/Test_Optimized.csv",  index=False)

print(f"Train rows: {split_idx}  |  Test rows: {len(df_full)-split_idx}")
print("Data ready.")


Train rows: 4022  |  Test rows: 1724
Data ready.


In [3]:
# ── CELL 3: ENVIRONMENT — FIXED REWARD ──────────────────────
# THE KEY FIX IS HERE.
#
# Old reward: -temp_error - 0.7*energy - smooth
#   All actions gave rewards in range [-0.1, -0.5] — nearly same.
#   MCTS had NO signal to prefer one action over another.
#
# New reward: adds ACTION-SPECIFIC SHAPED REWARDS
#   When room is comfortable AND energy is high AND we DECREASE:
#     → big bonus (+0.5): this is the RIGHT action
#   When room is uncomfortable AND we DECREASE:
#     → big penalty (-0.5): this is the WRONG action
#   When room is uncomfortable AND we INCREASE:
#     → bonus (+0.3): this is the RIGHT action
#
# This creates a 3-10x reward difference between good/bad actions,
# giving MCTS a CLEAR signal to prefer one branch over another.

STATE_SIZE  = 15
ACTION_SIZE = 3

class HVACEnv:
    def __init__(self, data_path, damper_step=0.025, lam=0.7,
                 comfort_threshold=0.15, energy_threshold=0.1):
        self.df                = pd.read_csv(data_path).reset_index(drop=True)
        self.damper_step       = damper_step
        self.lam               = lam
        self.comfort_threshold = comfort_threshold  # |temp_error| below = comfortable
        self.energy_threshold  = energy_threshold   # energy above = wasteful
        self.max_index         = len(self.df) - 1
        self.damper_idx        = self.df.columns.get_loc("damper_position")
        self.airflow_idx       = self.df.columns.get_loc("airflow_current")
        self.room_idx          = self.df.columns.get_loc("room_temp")
        self.setpoint_idx      = self.df.columns.get_loc("setpoint")
        self.signal_idx        = self.df.columns.get_loc("thermal_signal")
        self.prev_damper       = None
        self.last_action       = 1   # track last action for shaping
        self.reset()

    def reset(self, start_idx=None):
        if start_idx is not None:
            self.idx = int(start_idx)
        else:
            valid = np.where(self.df["damper_position"].values > 0)[0]
            self.idx = int(np.random.choice(valid))
        self.current_damper = float(self.df.iloc[self.idx]["damper_position"])
        self.prev_damper    = self.current_damper
        self.last_action    = 1
        return self._get_state()

    def _get_state(self):
        return self.df.iloc[self.idx].values.astype(np.float32)

    def step(self, action):
        self.prev_damper = self.current_damper
        self.last_action = action

        if action == 0: self.current_damper -= self.damper_step
        elif action == 2: self.current_damper += self.damper_step
        self.current_damper = np.clip(self.current_damper, 0.0, 1.0)

        self.idx += 1
        if self.idx >= self.max_index:
            self.idx = self.max_index
            done = True
        else:
            done = False

        row        = self.df.iloc[self.idx]
        base       = max(row["damper_position"], 1e-3)
        ratio      = np.clip(self.current_damper / base, 0.2, 2.0)
        airflow    = np.clip(row["airflow_current"] * ratio, 0.0, 1.0)
        energy     = airflow * row["thermal_signal"]
        temp_error = abs(row["room_temp"] - row["setpoint"])
        smooth     = 0.1 * abs(self.current_damper - self.prev_damper)

        # ── BASE REWARD (same as before) ──
        reward = -temp_error - self.lam * energy - smooth

        # ── ACTION SHAPING: creates clear contrast between actions ──
        comfortable = temp_error < self.comfort_threshold
        wasteful    = energy > self.energy_threshold

        if comfortable and wasteful and action == 0:
            # Room OK + energy high + we decrease → CORRECT → big bonus
            reward += 0.5
        elif comfortable and wasteful and action == 2:
            # Room OK + energy high + we increase → WRONG → big penalty
            reward -= 0.5
        elif (not comfortable) and action == 2:
            # Room uncomfortable + we increase → CORRECT → bonus
            reward += 0.3
        elif (not comfortable) and action == 0:
            # Room uncomfortable + we decrease → WRONG → penalty
            reward -= 0.3
        elif comfortable and (not wasteful) and action == 1:
            # Room OK + energy already low + we hold → CORRECT → small bonus
            reward += 0.1

        next_state                  = row.values.astype(np.float32).copy()
        next_state[self.damper_idx] = self.current_damper
        return next_state, reward, done, {"energy": energy, "temp_error": temp_error}


# Verify reward contrast
print("=" * 50)
print("REWARD CONTRAST VERIFICATION")
print("=" * 50)
env = HVACEnv("Dataset/Transformed_Optimized.csv")
# Find a wasteful + comfortable state
env.idx = 100
env.current_damper = 0.8
env.prev_damper    = 0.8
rewards = {}
for a in [0, 1, 2]:
    import copy
    e2 = copy.deepcopy(env)
    _, r, _, info = e2.step(a)
    rewards[a] = (r, info)
    print(f"  Action {['DECREASE','HOLD','INCREASE'][a]:8s}: reward={r:+.4f} "
          f"energy={info['energy']:.4f} temp_err={info['temp_error']:.4f}")
r_diff = rewards[0][0] - rewards[2][0]
print(f"  Reward contrast DECREASE vs INCREASE: {r_diff:+.4f}")
print(f"  (was near 0 before, needs to be > 0.3 for MCTS to learn)")
print("=" * 50)


REWARD CONTRAST VERIFICATION
  Action DECREASE: reward=-1.0611 energy=-0.0000 temp_err=0.7586
  Action HOLD    : reward=-0.7586 energy=-0.0000 temp_err=0.7586
  Action INCREASE: reward=-0.4611 energy=-0.0000 temp_err=0.7586
  Reward contrast DECREASE vs INCREASE: -0.6000
  (was near 0 before, needs to be > 0.3 for MCTS to learn)


In [4]:
# ── CELL 4: NEURAL NETWORK (unchanged) ──────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F

class AlphaThermalNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(STATE_SIZE, 128), nn.LayerNorm(128), nn.LeakyReLU(0.01), nn.Dropout(0.2),
            nn.Linear(128, 128),        nn.LayerNorm(128), nn.LeakyReLU(0.01), nn.Dropout(0.2),
            nn.Linear(128, 128),        nn.LayerNorm(128), nn.LeakyReLU(0.01), nn.Dropout(0.2),
        )
        self.policy_head = nn.Sequential(nn.Linear(128,64), nn.LeakyReLU(0.01), nn.Linear(64, ACTION_SIZE))
        self.value_head  = nn.Sequential(nn.Linear(128,64), nn.LeakyReLU(0.01), nn.Linear(64, 1))

    def forward(self, x):
        h = self.shared(x)
        return F.softmax(self.policy_head(h), dim=-1), torch.tanh(self.value_head(h))

net = AlphaThermalNet()
p1, v1 = net(torch.zeros(1, STATE_SIZE))
print("Network OK — batch=1:", p1.shape, v1.shape)


Network OK — batch=1: torch.Size([1, 3]) torch.Size([1, 1])


In [5]:
# ── CELL 5: MCTS (unchanged) ─────────────────────────────────
import copy

class MCTSNode:
    def __init__(self, state, parent=None, prior=1.0):
        self.state=state; self.parent=parent; self.children={}
        self.visits=0; self.value=0.0; self.prior=prior

class MCTS:
    def __init__(self, env, model, simulations=50, c_puct=1.5,
                 dirichlet_alpha=0.3, dirichlet_epsilon=0.25):
        self.env=env; self.model=model; self.simulations=simulations
        self.c_puct=c_puct; self.dirichlet_alpha=dirichlet_alpha
        self.dirichlet_epsilon=dirichlet_epsilon; self.actions=[0,1,2]

    def search(self, root_state, add_noise=True):
        root = MCTSNode(root_state)
        rp   = self._get_policy(root_state)
        if add_noise:
            noise = np.random.dirichlet([self.dirichlet_alpha]*3)
            rp = (1-self.dirichlet_epsilon)*rp + self.dirichlet_epsilon*noise
        er = copy.deepcopy(self.env)
        for a in self.actions:
            et = copy.deepcopy(er)
            ns,_,_,_ = et.step(a)
            root.children[a] = MCTSNode(ns, parent=root, prior=rp[a])
        for _ in range(self.simulations):
            ec = copy.deepcopy(self.env); node = root
            while node.children:
                action, node = self._select(node)
                _,_,done,_ = ec.step(action)
                if done: break
            if not node.children:
                pr = self._get_policy(ec._get_state())
                for a in self.actions:
                    et=copy.deepcopy(ec); ns,_,_,_=et.step(a)
                    node.children[a]=MCTSNode(ns,parent=node,prior=pr[a])
            self._backprop(node, self._evaluate(ec))
        visits = np.array([root.children[a].visits if a in root.children else 0
                           for a in self.actions], dtype=np.float32)
        s = visits.sum()
        return int(np.argmax(visits)), (visits/s if s>0 else np.ones(3)/3.0)

    def _select(self, node):
        best,ba,bc=-np.inf,None,None
        for a,c in node.children.items():
            q=c.value/(c.visits+1e-6)
            u=self.c_puct*c.prior*np.sqrt(node.visits+1)/(1+c.visits)
            if q+u>best: best,ba,bc=q+u,a,c
        return ba,bc

    def _get_policy(self, state):
        self.model.eval()
        with torch.no_grad():
            p,_=self.model(torch.tensor(state,dtype=torch.float32).unsqueeze(0))
        return p.squeeze(0).numpy()

    def _evaluate(self, ec):
        self.model.eval()
        with torch.no_grad():
            _,v=self.model(torch.tensor(ec._get_state(),dtype=torch.float32).unsqueeze(0))
        return v.item()

    def _backprop(self, node, value):
        while node:
            node.visits+=1; node.value+=value; node=node.parent

print("MCTS ready.")


MCTS ready.


In [6]:
# ── CELL 6: PRE-TRAINING ─────────────────────────────────────
print("Pre-training value head...")

train_env = HVACEnv("Dataset/Train_Optimized.csv")
model     = AlphaThermalNet()
start_idx = int(np.where(train_env.df["damper_position"].values > 0)[0][0])

pt_states, pt_rewards = [], []
s = train_env.reset(start_idx=start_idx)
for step in range(2000):
    pt_states.append(s.copy())
    row = train_env.df.iloc[train_env.idx]
    # Use the shaped reward signal for pre-training
    energy     = row["airflow_current"] * row["thermal_signal"]
    temp_error = abs(row["room_temp"] - row["setpoint"])
    comfortable = temp_error < 0.15
    wasteful    = energy > 0.1
    a = 0 if (comfortable and wasteful) else (2 if not comfortable else 1)
    s, r, done, _ = train_env.step(a)
    pt_rewards.append(r)
    if done: break

r_arr  = np.array(pt_rewards, dtype=np.float32)
r_norm = np.clip((r_arr - r_arr.mean()) / (r_arr.std() + 1e-8), -3, 3) / 3.0

pt_s = torch.tensor(np.array(pt_states), dtype=torch.float32)
pt_v = torch.tensor(r_norm,              dtype=torch.float32).unsqueeze(1)

pt_opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
model.train()
for epoch in range(100):
    perm = torch.randperm(len(pt_s))
    for i in range(0, len(pt_s), 256):
        idx = perm[i:i+256]
        _, pv = model(pt_s[idx])
        loss  = ((pv - pt_v[idx])**2).mean()
        pt_opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        pt_opt.step()
    if (epoch+1) % 25 == 0:
        print(f"  epoch {epoch+1}/100 | value_loss={loss.item():.5f}")

model.eval()
best_s  = pt_s[np.argmax(r_norm)].unsqueeze(0)
worst_s = pt_s[np.argmin(r_norm)].unsqueeze(0)
with torch.no_grad():
    _, vb = model(best_s)
    _, vw = model(worst_s)
print(f"  Value(best)={vb.item():.4f}  Value(worst)={vw.item():.4f}")
print(f"  {'OK — value head ranks correctly.' if vb.item()>vw.item() else 'WARNING — inverted.'}")


Pre-training value head...
  epoch 25/100 | value_loss=0.04710
  epoch 50/100 | value_loss=0.05139
  epoch 75/100 | value_loss=0.02282
  epoch 100/100 | value_loss=0.04896
  Value(best)=0.6486  Value(worst)=-0.8895
  OK — value head ranks correctly.


In [7]:
# ── CELL 7: TRAINING LOOP ────────────────────────────────────
NUM_ITERATIONS  = 10
NUM_EPISODES    = 20
STEPS_PER_EP    = 200
EPOCHS_PER_ITER = 150
BATCH_SIZE      = 256
LR              = 1e-3
GAMMA           = 0.99
VALUE_W         = 0.5
SIM_SCHEDULE    = [20, 20, 30, 30, 50, 50, 75, 75, 100, 100]
C_PUCT_SCHEDULE = np.linspace(2.0, 1.0, NUM_ITERATIONS)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)
max_start = max(start_idx+1, int(len(train_env.df)*0.85) - STEPS_PER_EP)

print(f"Iterations={NUM_ITERATIONS} | Episodes/iter={NUM_EPISODES} | Steps/ep={STEPS_PER_EP}")
print(f"WATCH: policy_loss should DROP below {np.log(3):.3f} (log3).")
print(f"If it stays at ~1.099, policy is still random.")
print()

for iteration in range(NUM_ITERATIONS):
    current_sims   = SIM_SCHEDULE[iteration]
    current_c_puct = float(C_PUCT_SCHEDULE[iteration])

    mcts = MCTS(train_env, model, simulations=current_sims,
                c_puct=current_c_puct)

    memory_states, memory_policies, memory_values = [], [], []

    for ep in range(NUM_EPISODES):
        rand_start = np.random.randint(start_idx, max_start)
        state      = train_env.reset(start_idx=rand_start)
        ep_states, ep_policies, ep_rewards, ep_nexts = [], [], [], []

        for step in range(STEPS_PER_EP):
            action, policy_dist = mcts.search(state, add_noise=True)
            next_state, reward, done, _ = train_env.step(action)
            ep_states.append(state); ep_policies.append(policy_dist)
            ep_rewards.append(reward); ep_nexts.append(next_state)
            state = next_state
            if done: break

        # TD targets
        model.eval()
        td_targets = []
        with torch.no_grad():
            for i in range(len(ep_states)):
                r = ep_rewards[i]
                if i+1 < len(ep_states):
                    ns_t = torch.tensor(ep_nexts[i], dtype=torch.float32).unsqueeze(0)
                    _, nv = model(ns_t)
                    td_targets.append(r + GAMMA * nv.item())
                else:
                    td_targets.append(r)
        model.train()

        memory_states.extend(ep_states); memory_policies.extend(ep_policies)
        memory_values.extend(td_targets)

    total_samples = len(memory_states)

    raw_v  = np.array(memory_values, dtype=np.float32)
    norm_v = np.clip((raw_v - raw_v.mean()) / (raw_v.std()+1e-8), -3, 3) / 3.0

    states_t   = torch.tensor(np.array(memory_states,   dtype=np.float32))
    policies_t = torch.tensor(np.array(memory_policies, dtype=np.float32))
    values_t   = torch.tensor(norm_v, dtype=torch.float32).unsqueeze(1)

    model.train()
    for epoch in range(EPOCHS_PER_ITER):
        perm = torch.randperm(total_samples)
        ep_pl, ep_vl, nb = 0.0, 0.0, 0
        for s in range(0, total_samples, BATCH_SIZE):
            idx = perm[s:s+BATCH_SIZE]
            pp, pv = model(states_t[idx])
            pl   = -(policies_t[idx] * torch.log(pp+1e-8)).sum(dim=1).mean()
            vl   = ((pv - values_t[idx])**2).mean()
            loss = pl + VALUE_W * vl
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_pl+=pl.item(); ep_vl+=vl.item(); nb+=1

        if (epoch+1) % 50 == 0:
            avg_p = ep_pl/nb
            flag  = "OK" if avg_p < np.log(3)-0.05 else "STILL RANDOM"
            print(f"    Epoch {epoch+1:3d}/{EPOCHS_PER_ITER} "
                  f"| policy_loss={avg_p:.4f} [{flag}] "
                  f"| value_loss={ep_vl/nb:.4f} "
                  f"| LR={scheduler.get_last_lr()[0]:.6f}")

    scheduler.step()
    print(f"  Iter {iteration+1}/{NUM_ITERATIONS} done | sims={current_sims}\n")

torch.save(model.state_dict(), "alphaHVAC_trained.pth")
print("Training complete.")


Iterations=10 | Episodes/iter=20 | Steps/ep=200
WATCH: policy_loss should DROP below 1.099 (log3).
If it stays at ~1.099, policy is still random.

    Epoch  50/150 | policy_loss=1.0908 [STILL RANDOM] | value_loss=0.0361 | LR=0.001000
    Epoch 100/150 | policy_loss=1.0905 [STILL RANDOM] | value_loss=0.0346 | LR=0.001000
    Epoch 150/150 | policy_loss=1.0902 [STILL RANDOM] | value_loss=0.0339 | LR=0.001000
  Iter 1/10 done | sims=20

    Epoch  50/150 | policy_loss=1.0936 [STILL RANDOM] | value_loss=0.0285 | LR=0.001000
    Epoch 100/150 | policy_loss=1.0931 [STILL RANDOM] | value_loss=0.0278 | LR=0.001000
    Epoch 150/150 | policy_loss=1.0925 [STILL RANDOM] | value_loss=0.0269 | LR=0.001000
  Iter 2/10 done | sims=20

    Epoch  50/150 | policy_loss=1.0944 [STILL RANDOM] | value_loss=0.0324 | LR=0.001000
    Epoch 100/150 | policy_loss=1.0936 [STILL RANDOM] | value_loss=0.0315 | LR=0.001000
    Epoch 150/150 | policy_loss=1.0934 [STILL RANDOM] | value_loss=0.0312 | LR=0.001000
  Ite

KeyboardInterrupt: 

In [ ]:
# ── CELL 8: EVALUATION + ACTION PLOT ────────────────────────
# The action plot shows WHAT the model actually decided at each
# timestep. If it is intelligent you will see:
#   - Action 0 (DECREASE) when energy is high and room is OK
#   - Action 2 (INCREASE) when room is uncomfortable
#   - Action 1 (HOLD) when already efficient
# If all actions are the same → still random.
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

test_env  = HVACEnv("Dataset/Test_Optimized.csv")
mcts_eval = MCTS(test_env, model, simulations=100, c_puct=1.0)
state     = test_env.reset(start_idx=0)
model.eval()

baseline_energy,     model_energy      = [], []
baseline_temp_error, model_temp_error  = [], []
model_damper,        actions_taken     = [], []

for step in range(len(test_env.df) - 1):
    row = test_env.df.iloc[test_env.idx]
    baseline_energy.append(row["airflow_current"] * row["thermal_signal"])
    baseline_temp_error.append(abs(row["room_temp"] - row["setpoint"]))

    action, _ = mcts_eval.search(state, add_noise=False)
    actions_taken.append(action)

    state, _, done, info = test_env.step(action)
    model_energy.append(info["energy"])
    model_damper.append(test_env.current_damper)
    model_temp_error.append(abs(
        test_env.df.iloc[test_env.idx]["room_temp"] -
        test_env.df.iloc[test_env.idx]["setpoint"]
    ))
    if done: break

actions_arr = np.array(actions_taken)
n_dec  = (actions_arr == 0).sum()
n_hold = (actions_arr == 1).sum()
n_inc  = (actions_arr == 2).sum()
total  = len(actions_arr)

avg_base_e  = np.mean(baseline_energy)
avg_model_e = np.mean(model_energy)
avg_base_t  = np.mean(baseline_temp_error)
avg_model_t = np.mean(model_temp_error)
energy_saving = (1 - avg_model_e / avg_base_e) * 100
temp_change   = ((avg_model_t - avg_base_t) / (avg_base_t + 1e-8)) * 100

# ── FIGURE: 4 panels ──
fig, axes = plt.subplots(4, 1, figsize=(14, 16))
fig.suptitle("AlphaHVAC — Full Test Evaluation", fontsize=14, fontweight="bold")
steps = range(len(model_energy))

# Panel 1: Energy
axes[0].plot(baseline_energy, label="Baseline HVAC",  color="steelblue",  lw=1.0)
axes[0].plot(model_energy,    label="AlphaThermal",   color="darkorange", lw=1.0)
axes[0].fill_between(steps, baseline_energy, model_energy,
                     where=[b>m for b,m in zip(baseline_energy,model_energy)],
                     alpha=0.15, color="green", label="Saved")
axes[0].set_title(f"Energy — Saving: {energy_saving:.1f}%  "
                  f"(Baseline avg: {avg_base_e:.4f}  Model avg: {avg_model_e:.4f})")
axes[0].set_ylabel("Energy"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# Panel 2: Damper position over time
axes[1].plot(model_damper, color="purple", lw=1.0, label="Model damper position")
axes[1].axhline(0, color="red", linestyle="--", lw=0.8, label="Damper=0 (closed)")
axes[1].set_title("Model Damper Position Over Time")
axes[1].set_ylabel("Damper (0–1)"); axes[1].set_ylim(-0.05, 1.05)
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

# Panel 3: Actions taken at each step
# Color code: red=decrease, orange=hold, green=increase
action_colors_map = {0: "#e74c3c", 1: "#f39c12", 2: "#27ae60"}
colors = [action_colors_map[a] for a in actions_taken]
axes[2].scatter(steps, actions_taken, c=colors, s=3, alpha=0.6)
axes[2].set_yticks([0, 1, 2])
axes[2].set_yticklabels(["0: DECREASE", "1: HOLD", "2: INCREASE"])
axes[2].set_title(
    f"Actions at Each Step — "
    f"DECREASE: {n_dec} ({n_dec/total*100:.1f}%)  "
    f"HOLD: {n_hold} ({n_hold/total*100:.1f}%)  "
    f"INCREASE: {n_inc} ({n_inc/total*100:.1f}%)"
)
patches = [mpatches.Patch(color=action_colors_map[i],
           label=["DECREASE","HOLD","INCREASE"][i]) for i in [0,1,2]]
axes[2].legend(handles=patches, fontsize=8, loc="upper right")
axes[2].grid(alpha=0.3)

# Panel 4: Temperature error
axes[3].plot(baseline_temp_error, label="Baseline",      color="steelblue",  lw=1.0)
axes[3].plot(model_temp_error,    label="AlphaThermal",  color="darkorange", lw=1.0)
axes[3].set_title(f"Temperature Error — Change vs baseline: {temp_change:+.1f}%  "
                  f"(Baseline avg: {avg_base_t:.4f}  Model avg: {avg_model_t:.4f})")
axes[3].set_ylabel("|temp − setpoint|"); axes[3].set_xlabel("Timestep")
axes[3].legend(fontsize=8); axes[3].grid(alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("AlphaHVAC_Full_Eval.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to AlphaHVAC_Full_Eval.png")


In [ ]:
# ── CELL 9: PASS/FAIL REPORT ─────────────────────────────────
print("=" * 60)
print("AlphaHVAC — EVALUATION REPORT")
print("=" * 60)
print()

# A: Energy saving
grade_e = "GOOD" if energy_saving>=20 else ("WEAK" if energy_saving>=5 else "FAIL")
print(f"[A] ENERGY SAVING: {energy_saving:.1f}% — {grade_e}")
print(f"    Baseline: {avg_base_e:.4f}  |  Model: {avg_model_e:.4f}")
print()

# B: Comfort
grade_t = "GOOD" if temp_change<=5 else ("ACCEPTABLE" if temp_change<=20 else "POOR")
print(f"[B] THERMAL COMFORT: {temp_change:+.1f}% change — {grade_t}")
print(f"    Baseline: {avg_base_t:.4f}  |  Model: {avg_model_t:.4f}")
print()

# C: Action diversity — is the model NOT random?
action_entropy = -sum((c/total)*np.log(c/total+1e-8)
                      for c in [n_dec, n_hold, n_inc] if c > 0)
max_entropy    = np.log(3)
diversity_pct  = action_entropy / max_entropy * 100
print(f"[C] ACTION DIVERSITY (is policy random or decisive?)")
print(f"    DECREASE: {n_dec:4d} ({n_dec/total*100:.1f}%)")
print(f"    HOLD:     {n_hold:4d} ({n_hold/total*100:.1f}%)")
print(f"    INCREASE: {n_inc:4d} ({n_inc/total*100:.1f}%)")
print(f"    Action entropy: {action_entropy:.4f} / {max_entropy:.4f} = {diversity_pct:.1f}% of max")
if abs(n_dec/total - 1/3) < 0.03 and abs(n_inc/total - 1/3) < 0.03:
    grade_c = "FAIL — actions are near-uniform (random policy)"
elif max(n_dec, n_hold, n_inc)/total > 0.6:
    grade_c = "WEAK — one action dominates (biased but not intelligent)"
else:
    grade_c = "GOOD — diverse, context-dependent actions"
print(f"    Grade: {grade_c}")
print()

# D: Damper diagnostic
min_damp  = min(model_damper)
mean_damp = np.mean(model_damper)
frac_zero = sum(1 for d in model_damper if d < 0.01) / len(model_damper)
print(f"[D] DAMPER DIAGNOSTIC")
print(f"    Min damper  : {min_damp:.4f}")
print(f"    Mean damper : {mean_damp:.4f}")
print(f"    % time at 0 : {frac_zero*100:.1f}%")
if frac_zero > 0.5:
    grade_d = "FAIL — damper stuck at zero. Orange line=0 is NOT energy saving."
elif frac_zero > 0.2:
    grade_d = "WEAK — damper frequently at zero. Likely biased DECREASE."
else:
    grade_d = "GOOD — damper moves dynamically."
print(f"    Grade: {grade_d}")
print()

# Policy loss check
print(f"[E] POLICY LOSS (check from training output)")
print(f"    log(3) = {np.log(3):.4f} = random policy benchmark")
print(f"    If your final policy_loss was ~1.09, policy = still random.")
print(f"    If policy_loss < 1.00, policy has started learning.")
print(f"    If policy_loss < 0.80, policy is genuinely learning.")
print()

# Overall
print("=" * 60)
print("OVERALL")
print("=" * 60)
grades = [grade_e, grade_t, grade_c, grade_d]
n_good = sum(1 for g in grades if g.startswith("GOOD"))
n_fail = sum(1 for g in grades if g.startswith("FAIL"))

if n_fail == 0 and n_good >= 3:
    print("MODEL IS GENUINELY GOOD.")
    print("Energy saved, comfort maintained, policy is decisive.")
elif n_fail >= 2 and "FAIL — damper stuck" in grade_d:
    print("MODEL HAS NOT LEARNED — DAMPER STUCK AT ZERO.")
    print("The 65% energy saving is FALSE — it is just a closed damper.")
    print("The reward shaping fix (Cell 3) needs more iterations.")
    print("Try increasing NUM_ITERATIONS to 15 and NUM_EPISODES to 30.")
else:
    print("MODEL IS PARTIALLY WORKING.")
    print("Check [C] action diversity and [D] damper diagnostic above.")
print("=" * 60)
